# FIFA World Cup 2026 — Data Scraping NotebookThis notebook demonstrates how we collected player statistics directly from [FIFA's official website](https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/statistics/player-statistics) using **Selenium** for automated browser scraping.---## Workflow Overview1. **Open the FIFA statistics page** using a Selenium-controlled Chrome browser.2. **Navigate to the desired stat category** (e.g., Attacking, Defending, Discipline).3. **Click "Load More"** repeatedly until all rows are loaded into the DOM.4. **Scrape the table headers** and row data from the rendered HTML.5. **Export to CSV** in the `data/raw/` directory.> **Note:** The `data/raw/` CSV files are already included in this repository. Re-running this notebook will re-scrape from the live website, which may yield updated data.---## Prerequisites```bashpip install selenium```You will also need a Chrome browser installed. Selenium's built-in `webdriver-manager` will handle the ChromeDriver automatically.

In [ ]:
import timefrom selenium import webdriverfrom selenium.webdriver.common.by import Byfrom selenium.webdriver.support.ui import WebDriverWaitfrom selenium.webdriver.support import expected_conditions as ECimport pandas as pd

### Step 1: Launch Browser and Navigate to FIFA Stats Page

In [ ]:
# Initialize a headless-capable Chrome driverdriver = webdriver.Chrome()

In [ ]:
# Define the base URL for FIFA's player statistics page# Change the URL path to target different stat categories if neededbase_url = (    "https://www.fifa.com/en/tournaments/mens/worldcup/"    "canadamexicousa2026/statistics/player-statistics")

In [ ]:
# Navigate to the page and wait for initial content to loaddriver.get(base_url)# Allow the page to render dynamic contenttime.sleep(5)

### Step 2: Load All Rows via "Load More" Button

In [ ]:
# Set up an explicit wait (up to 20 seconds per element)wait = WebDriverWait(driver, 20)# Wait for the "Load More" button to appearbtn = wait.until(    EC.element_to_be_clickable((By.CLASS_NAME, 'button-label')))

In [ ]:
# Click "Load More" repeatedly until the button disappears (all rows loaded)while True:    try:        btn.click()        time.sleep(2)  # Allow new rows to render before next click    except Exception:        # Button no longer exists — all data is loaded        breakprint("All rows loaded successfully.")

### Step 3: Scrape Table Headers

In [ ]:
# Extract column headers from the table <thead>scraped_columns = driver.find_elements(    By.CSS_SELECTOR, "thead tr")# Split header text into individual column namescolumn_names = scraped_columns[0].text.split('\n')# Insert missing columns that appear in the data but not in the headercolumn_names.insert(2, 'Country')column_names.insert(3, 'Position')print(f"Column names ({len(column_names)}): {column_names}")

### Step 4: Scrape Table Rows

In [ ]:
# Extract all data rows from the table <tbody>scraped_rows = driver.find_elements(    By.CSS_SELECTOR, "tbody tr")# Parse each row into a list of valuesdata = []for row_element in scraped_rows:    row_values = row_element.text.split('\n')    data.append(row_values)print(f"Total rows scraped: {len(data)}")

### Step 5: Export to CSV

In [ ]:
# Set the category name to match the stat page you scraped# Options: "Attacking", "Defending", "Discipline", "Distribution", "Goalkeeping", "Golden_Boot"category_name = "Attacking"# Create a DataFrame and save to data/raw/df = pd.DataFrame(data, columns=column_names)df.to_csv(f"../data/raw/{category_name}.csv", index=False)print(f"Saved {len(df)} rows to ../data/raw/{category_name}.csv")

### Step 6: Close Browser

In [ ]:
# Always close the browser to free resourcesdriver.quit()print("Browser closed.")

---## Repeating for All CategoriesTo scrape all six categories, repeat Steps 1–6 for each URL:| Category | URL Suffix ||----------|-----------|| Attacking | `/attacking` || Defending | `/defending` || Discipline | `/discipline` || Distribution | `/distribution` || Goalkeeping | `/goalkeeping` || Golden Boot | `/golden-boot` |After scraping all categories, proceed to the **EDA Analysis** notebook (`02_eda_analysis.ipynb`).